# 🔍 Day 03: Graph DFS — Components, Cycles & Backtracking

---

## DFS on Graphs — Go Deep, Then Backtrack

DFS explores as far as possible along a branch before backtracking. On graphs, it's perfect for:
- Finding connected components
- Detecting cycles
- Path finding with constraints
- Flood fill / region marking

---

In [ ]:
from collections import defaultdict, deque

## 🎯 Problem 1: Course Schedule — Cycle Detection (LC #207)

Given `numCourses` and prerequisites `[a, b]` (take b before a), determine if you can finish all courses. Basically: is there a cycle in the dependency graph?

**DFS cycle detection in directed graph:** Use 3 states:
- `unvisited`: Not yet explored
- `in_progress`: Currently in DFS path (if we revisit → cycle!)
- `completed`: Fully explored, no cycle through this node

In [ ]:
def canFinish(numCourses, prerequisites):
    """
    DFS cycle detection with 3 states.
    Time: O(V + E), Space: O(V + E)
    """
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)
    
    # 0 = unvisited, 1 = in progress, 2 = completed
    state = [0] * numCourses
    
    def has_cycle(node):
        if state[node] == 1:  # Back edge → cycle!
            return True
        if state[node] == 2:  # Already fully explored
            return False
        
        state[node] = 1  # Mark in progress
        for neighbor in graph[node]:
            if has_cycle(neighbor):
                return True
        state[node] = 2  # Mark completed
        return False
    
    for course in range(numCourses):
        if has_cycle(course):
            return False
    return True

print(canFinish(2, [[1,0]]))          # True (take 0 then 1)
print(canFinish(2, [[1,0],[0,1]]))    # False (cycle!)

**Why 3 states instead of 2?** With just visited/unvisited, you can't tell if you're revisiting a node in the current DFS path (cycle) vs a node fully processed from a different path (no cycle).

---

## 🎯 Problem 2: Pacific Atlantic Water Flow (LC #417)

Given a height matrix, find cells where water can flow to BOTH the Pacific (top/left edges) and Atlantic (bottom/right edges).

**Reverse thinking:** Instead of flowing FROM each cell, DFS FROM the ocean edges inward. Find cells reachable from Pacific, cells reachable from Atlantic, return the intersection.

In [ ]:
def pacificAtlantic(heights):
    """
    DFS from both ocean edges, find intersection.
    Time: O(m*n), Space: O(m*n)
    """
    if not heights:
        return []
    rows, cols = len(heights), len(heights[0])
    pacific = set()
    atlantic = set()
    
    def dfs(r, c, reachable, prev_height):
        if (r, c) in reachable:
            return
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return
        if heights[r][c] < prev_height:  # Water flows downhill
            return
        
        reachable.add((r, c))
        for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
            dfs(r+dr, c+dc, reachable, heights[r][c])
    
    # DFS from Pacific edges (top row + left col)
    for c in range(cols):
        dfs(0, c, pacific, 0)
        dfs(rows-1, c, atlantic, 0)
    for r in range(rows):
        dfs(r, 0, pacific, 0)
        dfs(r, cols-1, atlantic, 0)
    
    return list(pacific & atlantic)

heights = [
    [1,2,2,3,5],
    [3,2,3,4,4],
    [2,4,5,3,1],
    [6,7,1,4,5],
    [5,1,1,2,4]
]
result = pacificAtlantic(heights)
print(f"Cells reaching both oceans: {sorted(result)}")

---

## 🎯 Problem 3: Surrounded Regions (LC #130)

Capture all `'O'` regions surrounded by `'X'`. An `'O'` on the border can't be captured.

**Reverse thinking again:** DFS from border `'O'`s, mark them safe. Everything else gets captured.

In [ ]:
def solve(board):
    """
    Mark border-connected O's as safe, flip the rest.
    Time: O(m*n), Space: O(m*n)
    """
    if not board:
        return
    rows, cols = len(board), len(board[0])
    
    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return
        if board[r][c] != 'O':
            return
        board[r][c] = 'S'  # Safe (border-connected)
        for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
            dfs(r+dr, c+dc)
    
    # Mark border O's as safe
    for r in range(rows):
        dfs(r, 0)
        dfs(r, cols - 1)
    for c in range(cols):
        dfs(0, c)
        dfs(rows - 1, c)
    
    # Flip: O → X (captured), S → O (restore safe)
    for r in range(rows):
        for c in range(cols):
            if board[r][c] == 'O':
                board[r][c] = 'X'
            elif board[r][c] == 'S':
                board[r][c] = 'O'

board = [
    ['X','X','X','X'],
    ['X','O','O','X'],
    ['X','X','O','X'],
    ['X','O','X','X']
]
solve(board)
for row in board:
    print(row)

---

## 🗺️ Graph DFS Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  GRAPH DFS PATTERNS                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. CYCLE DETECTION (directed): 3 states (unvisited/progress/done) ║
║     Back edge (revisit in-progress node) = cycle                    ║
║                                                                      ║
║  2. BOUNDARY DFS: Start from edges, mark safe, flip the rest       ║
║     Surrounded regions, Pacific Atlantic                            ║
║                                                                      ║
║  3. CONNECTED COMPONENTS: DFS from each unvisited, count starts    ║
║     Number of islands, provinces                                    ║
║                                                                      ║
║  TRICK: "Reverse thinking" — DFS from the answer backwards         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 207 | Course Schedule | Medium | Cycle detection, 3 states |
| 417 | Pacific Atlantic Water Flow | Medium | Boundary DFS, intersection |
| 130 | Surrounded Regions | Medium | Boundary DFS, mark safe |
| 547 | Number of Provinces | Medium | Connected components |
| 695 | Max Area of Island | Medium | DFS, track area |
| 733 | Flood Fill | Easy | DFS from source |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Directed cycle detection: 3 states, not 2                      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Boundary DFS: start from edges, think in reverse               │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Components: count DFS starts on unvisited nodes                │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Grid DFS: modify in-place to mark visited when possible        │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Topological Sort

Ordering dependencies — the Course Schedule II algorithm.